# Q08: Implement Self-Attention in NumPy
**Topic:** Coding | **Difficulty:** Hard | **Time:** 30 min
**Sheet:** GrindLLM50

---

## Question
Implement Self-Attention in NumPy.

## Detailed Answer

### Self-Attention Mechanism
Given input $X \in \mathbb{R}^{n \times d}$:
1. Project to Q, K, V: $Q = XW_Q$, $K = XW_K$, $V = XW_V$
2. Compute attention scores: $A = \text{softmax}(QK^T / \sqrt{d_k})$
3. Weighted values: $\text{output} = AV$

### Multi-Head Extension
Split Q, K, V into $h$ heads, compute attention per head, concatenate, project.

In [ ]:
import numpy as np

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax."""
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)


def scaled_dot_product_attention(
    Q: np.ndarray, K: np.ndarray, V: np.ndarray,
    mask: np.ndarray | None = None
) -> tuple[np.ndarray, np.ndarray]:
    """Scaled dot-product attention.
    
    Args:
        Q: Query matrix (seq_len, d_k)
        K: Key matrix (seq_len, d_k)
        V: Value matrix (seq_len, d_v)
        mask: Optional boolean mask (seq_len, seq_len), True = mask out
    Returns:
        output: Weighted values (seq_len, d_v)
        attention_weights: (seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)  # (n, n)
    
    if mask is not None:
        scores = np.where(mask, -1e9, scores)
    
    attn_weights = softmax(scores, axis=-1)  # (n, n)
    output = attn_weights @ V  # (n, d_v)
    return output, attn_weights


def self_attention(
    X: np.ndarray, d_model: int, d_k: int, d_v: int,
    W_Q: np.ndarray, W_K: np.ndarray, W_V: np.ndarray,
    mask: np.ndarray | None = None
) -> tuple[np.ndarray, np.ndarray]:
    """Single-head self-attention.
    
    Args:
        X: Input embeddings (seq_len, d_model)
        d_model: Embedding dimension
        d_k: Key/Query dimension
        d_v: Value dimension
        W_Q, W_K, W_V: Projection matrices
        mask: Optional causal mask
    """
    Q = X @ W_Q  # (n, d_k)
    K = X @ W_K  # (n, d_k)
    V = X @ W_V  # (n, d_v)
    return scaled_dot_product_attention(Q, K, V, mask)


def multi_head_attention(
    X: np.ndarray, n_heads: int, d_model: int,
    W_Qs: list, W_Ks: list, W_Vs: list, W_O: np.ndarray,
    mask: np.ndarray | None = None
) -> np.ndarray:
    """Multi-head self-attention."""
    d_k = d_model // n_heads
    heads = []
    for i in range(n_heads):
        head_out, _ = self_attention(X, d_model, d_k, d_k, W_Qs[i], W_Ks[i], W_Vs[i], mask)
        heads.append(head_out)
    concat = np.concatenate(heads, axis=-1)  # (n, d_model)
    return concat @ W_O  # (n, d_model)


# --- Demo ---
np.random.seed(42)
seq_len, d_model = 4, 8
X = np.random.randn(seq_len, d_model)

# Single-head
W_Q = np.random.randn(d_model, d_model) * 0.1
W_K = np.random.randn(d_model, d_model) * 0.1
W_V = np.random.randn(d_model, d_model) * 0.1

# Causal mask (upper triangle = True = masked)
causal_mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
print('Causal mask:\n', causal_mask.astype(int))

output, attn = self_attention(X, d_model, d_model, d_model, W_Q, W_K, W_V, mask=causal_mask)
print(f'\nInput shape: {X.shape}')
print(f'Output shape: {output.shape}')
print(f'Attention weights:\n{np.round(attn, 3)}')
print(f'Attention rows sum to 1: {np.allclose(attn.sum(axis=-1), 1.0)}')

# Multi-head
n_heads = 2
d_k = d_model // n_heads
W_Qs = [np.random.randn(d_model, d_k) * 0.1 for _ in range(n_heads)]
W_Ks = [np.random.randn(d_model, d_k) * 0.1 for _ in range(n_heads)]
W_Vs = [np.random.randn(d_model, d_k) * 0.1 for _ in range(n_heads)]
W_O = np.random.randn(d_model, d_model) * 0.1

mha_output = multi_head_attention(X, n_heads, d_model, W_Qs, W_Ks, W_Vs, W_O, causal_mask)
print(f'\nMHA output shape: {mha_output.shape}')

## Key Takeaways
- Self-attention: Q, K, V projections → scaled dot product → softmax → weighted V
- **Causal mask** fills upper triangle with -inf before softmax
- Multi-head = multiple parallel attention heads → concatenate → project
- Scaling by $\sqrt{d_k}$ prevents softmax saturation for large d